# R03 — mini_7 real-cloud replay and temporal development demonstration

This notebook reads the sealed R03 run. Raw PCDs, causal replay, overlays, fitting and state updates live in importable `valid_point` modules. mini_7 is **one development segment**; plots and any apparent effects are illustrative. Alarms are null when clean calibration support is below the unchanged 200-known rule. Interval marks are evaluator-only and loaded after the prediction seal.

The benign adapter control applies +0.05 m translation, +0.2° yaw and delays modeled pose availability by +10 ms in memory. Original cloud/source timestamps and odometry files remain unchanged. This combines calibration and timing uncertainty; it does not measure real clock error.

In [ ]:
import os, json, csv
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
run=Path(os.environ['VP_RUN_DIR'])
config=json.loads((run/'config.json').read_text())
seal=json.loads((run/'prediction_seal.json').read_text())
intervals=json.loads((run/'evaluator_intervals.json').read_text())
assert intervals['prediction_seal_sha256']
print('Run:',run.name,'Segment:',config['segment'],'units: seconds, metres, points/cell/frame; ideal arrival=source')
print('Reference:',config['fit'],'calibration:',config['diagnostic_calibration'])
print('Predictions sealed before evaluator intervals:',len(seal),'files')

In [ ]:
def table(path):
    with open(path,newline='') as f: return list(csv.DictReader(f))
def number(x):
    try: return float(x)
    except (ValueError,TypeError): return np.nan
attempts=table(run/'T_R03_attempts.csv')
roundtrip=table(run/'T_R03_roundtrip.csv')
checks=table(run/'T_R03_state_checks.csv')
flows=table(run/'T_R03_episode_flow.csv')
refs=table(run/'T_R03_references.csv')
cal=table(run/'T_R03_calibration.csv')
print('Derived attempts:',len(attempts),'accepted:',sum(r['reader_status']=='accepted' for r in attempts),'zero realized:',sum(r['realized_changed_rows']=='0' for r in attempts))
print('Identity controls / max field error:',[(r['family'],r['max_abs_field_error']) for r in roundtrip])
print('Exploratory reference status:',{s:sum(r['status']==s for r in refs) for s in set(r['status'] for r in refs)})
print('Calibration rows:',[(r['stream'],r['arm'],r['instant_known'],r['temporal_ready_known'],r['status']) for r in cal])
print('Technical checks:',checks)
display(Markdown((run/'gate.md').read_text()))

In [ ]:
# Fixed display choice: first configured sustained family and sync 75.
from valid_point.io.mixed_signals import parse_cloud_name, raw_to_top, transform_xyz
from valid_point.replay import AllowedSource, ReplayConfig, _select_pose
first=flows[0]
cloud=np.load(run/(first['episode']+'_geometry.npz'))
a,b=cloud['clean'],cloud['derived']
repair=np.load(run/'F_R03_corrected_rearrangement_example.npz')
target=json.loads((run/'target.json').read_text())
name=parse_cloud_name(str(cloud['cloud_id']))
rawroot=Path('data/mixed_signals')/config['revision']/'raw'
source=AllowedSource(rawroot/'PointClouds/mini_7',rawroot/'Odometry/mini_7')
pose,pose_reason=_select_pose(source.read_poses('003'),name.source_ns,name.source_ns,ReplayConfig().max_pose_age_ns)
assert pose_reason=='VALID'
mat=raw_to_top('003',pose.matrix)
views=[a,b,repair['derived']]
common=[transform_xyz(data[:,:3],mat) for data in views]
center_native=np.array([target['center_xyz_m']]) if target['center_xyz_m'] else None
center_top=transform_xyz(center_native,mat)[0] if center_native is not None else None
failed=repair['failed_ids']
failed_top=transform_xyz(repair['clean'][failed,:3],mat)
fig,axes=plt.subplots(2,3,figsize=(17,10),sharex='row',sharey='row')
titles=['Clean recorded cloud','Derived '+first['family'],'Corrected rearrangement; failed rows remain']
for col,title in enumerate(titles):
    for row,data in enumerate((views[col],common[col])):
        axis=axes[row,col]
        view=data[::max(1,len(data)//5000)]
        axis.scatter(view[:,0],view[:,1],s=.35,alpha=.5,color='#2463a0')
        center=center_native[0] if row==0 and center_native is not None else center_top
        if center is not None: axis.plot(center[0],center[1],'r+',ms=10)
        axis.set(xlabel=('Native x (m)' if row==0 else 'Top-frame x (m)'),
                 ylabel=('Native y (m)' if row==0 else 'Top-frame y (m)'),
                 title=f'{title}; N={len(data):,}' if row==0 else 'Same cloud in common top frame')
        axis.set_aspect('equal'); axis.grid(alpha=.2)
axes[0,2].scatter(repair['clean'][failed,0],repair['clean'][failed,1],s=20,c='red',marker='x',label=f'{len(failed)} unchanged failed proposals')
axes[1,2].scatter(failed_top[:,0],failed_top[:,1],s=20,c='red',marker='x')
axes[0,2].legend(fontsize=7,loc='lower left')
fig.suptitle('F_R03_geometry — native and top-frame views; identical bounds per row; mini_7 development only')
fig.tight_layout(rect=(0,.05,1,.93))
fig.subplots_adjust(hspace=.30)
for ext in ('png','svg','pdf'):
    fig.savefig(run/('F_R03_geometry.'+ext),dpi=170,bbox_inches='tight')
    fig.savefig(run/('F_R03_geometry_mini_7_development_'+run.name+'.'+ext),dpi=170,bbox_inches='tight')
display(fig)
plt.close(fig)

In [ ]:
def score_rows(episode):
    name='clean_scores.csv' if episode=='clean' else episode+'_scores.csv'
    return table(run/'sealed_predictions'/name)
clean=score_rows('clean')
selected=[first, next(r for r in flows if r['family']=='displacement' and r['schedule']=='burst'), next(r for r in flows if r['family']=='noise'), next(r for r in flows if r['family']=='calibration_timing')]
fig,axes=plt.subplots(7,1,figsize=(14,18),sharex=True)
colors=['#162d50','#c33b32','#d28a23','#3b8273','#7e54a1']
def plot_episode(rows,label,color,stream='003',arm='D-only',alpha=.9):
    v=[r for r in rows if r['stream']==stream and r['arm']==arm]
    t=np.array([number(r['elapsed_s']) for r in v])
    y=lambda key: np.array([number(r[key]) for r in v])
    for axis,key in zip(axes[:6],['raw_D_inside_points','raw_G_m','D_anomaly','G_anomaly','C','C_time']): axis.plot(t,y(key),label=label,color=color,alpha=alpha,lw=1)
plot_episode(clean,'clean 003',colors[0])
for i,flow in enumerate(selected): plot_episode(score_rows(flow['episode']),flow['family']+' '+flow['schedule'],colors[i+1],alpha=.8)
for stream,style in [('004','-'),('laser','--'),('top',':'),('dome','-.')]:
    v=[r for r in clean if r['stream']==stream and r['arm']=='D-only']
    axes[6].plot([number(r['elapsed_s']) for r in v],[number(r['C']) for r in v],style,lw=1,label=stream+' ('+('RSU' if stream in ('top','dome') else stream)+')')
for i,label in enumerate(['D inside points (count/frame)','G directed novelty (m)','D normalized anomaly [0,1]','G normalized anomaly [0,1]','Memoryless C [0,1]','Temporal C_time [0,1]','Honest companion C [0,1]']):
    axes[i].set_ylabel(label); axes[i].grid(alpha=.2); axes[i].legend(loc='best',fontsize=7,ncol=3)
for start,end in config['schedules_s']['sustained']:
    for axis in axes[:6]: axis.axvspan(start,end,color='red',alpha=.045)
for start,end in config['schedules_s']['burst']:
    for axis in axes[:6]: axis.axvspan(start,end,color='blue',alpha=.04)
axes[-1].set_xlabel('Elapsed acquisition time (s)')
fig.suptitle('F_R03_timelines — development only; red sustained / blue burst evaluator shading joined after seal; alarms unavailable',y=.995)
fig.tight_layout()
for ext in ('png','svg','pdf'):
    fig.savefig(run/('F_R03_timelines.'+ext),dpi=170)
    fig.savefig(run/('F_R03_timelines_mini_7_development_'+run.name+'.'+ext),dpi=170)
display(fig)
plt.close(fig)

In [ ]:
# Raw/normalized records and missingness are shown without replacing nulls by zero.
for flow in selected[:2]:
    rows=score_rows(flow['episode'])
    sample=[r for r in rows if r['stream']=='003' and r['arm']=='G-only' and 5<=number(r['elapsed_s'])<5.5]
    print(flow['family'],flow['schedule'],'G-only rows at onset:',[(r['elapsed_s'],r['raw_G_m'],r['G_reason'],r['A'],r['Z'],r['state_reason'],r['temporal_alarm']) for r in sample])
print('History contamination is visible by comparing post-offset G traces; it is not treated as guaranteed recovery.')
print('All alarms:',set(r['instant_alarm'] for r in clean),'temporal:',set(r['temporal_alarm'] for r in clean))
print('No C0–C3 or H evidence is present in this run.')
failed=[(r['cloud_id'],r['family'],r['failed_count'],r['injection_reason'],r['reader_status']) for r in attempts if number(r['failed_count'])>0 or r['reader_status']!='accepted' or number(r['realized_changed_rows'])==0]
print('Failed/zero-realized/rejected attempts (first 15 of full ledger):',failed[:15],'; total',len(failed))
for flow in selected[:2]:
    companion=[r for r in score_rows(flow['episode']) if r['stream']!='003' and r['arm']=='D-only']
    baseline=[r for r in clean if r['stream']!='003' and r['arm']=='D-only']
    paired=all((a['C'],a['C_time'],a['A'],a['Z'])==(b['C'],b['C_time'],b['A'],b['Z']) for a,b in zip(companion,baseline))
    print('Honest companion score/state equality for',flow['family'],':',paired,'rows',len(companion))

history=table(run/'T_R03_history.csv')
print('First second after the last intervention offset (paired known G; mean absolute difference in metres):')
for r in history:
    if r['phase']=='post_first_1s': print(r['family'],r['schedule'],r['paired_known_G'],r['mean_abs_paired_G_difference_m'])

for name in ('normalized_factors','instantaneous_decisions','temporal_states','alarm_decisions'):
    records=table(run/('T_R03_'+name+'.csv'))
    first_record=records[0]
    keys={'normalized_factors':('cloud_id','D','G','D_reason','G_reason'),
          'instantaneous_decisions':('cloud_id','arm','status','A','C'),
          'temporal_states':('cloud_id','arm','status','Z','C_time','ready'),
          'alarm_decisions':('cloud_id','arm','status','instant_alarm','temporal_alarm','reason')}[name]
    print('Distinct',name,'records:',len(records),'example:',{k:first_record[k] for k in keys})
